# Transformer Sequence Classification Practice

**Topic:** Attention and Transformers Review &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(6000, 20)`, `y_train`: integer token sequences (vocabulary 0–9), and label 1 iff token `7` occurs ≥ 3 times.
- `X_test` `(1500, 20)`: predict these. `y_test` is used by the evaluation cell.
- Positives and negatives are balanced, and near-miss sequences (exactly 2 or exactly 3 sevens) are common, so the decision boundary is genuinely sharp.

## Your task

1. Build a classifier for these sequences. Any model is allowed: a PyTorch embedding layer with one or two `nn.TransformerEncoderLayer`s and a binary head is the intended route, and embedding + mean pooling + MLP or a hand-built count feature also work.
2. Train with Adam a few epochs.
3. Set `predictions` to your 0/1 labels for `X_test`.

(A hand-built "count the sevens" feature also passes, and the reference solution below takes that route. Build the transformer from step 1 as well and check it reaches the same accuracy.)

You will need `torch` for the transformer route and `scikit-learn` for the reference solution. Both run on a laptop CPU in seconds.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.integers(0, 10, (n, 20))
    # force lots of near-boundary examples (2 or 3 sevens)
    counts = r.choice([0, 1, 2, 3, 4, 5], size=n, p=[0.1, 0.15, 0.25, 0.25, 0.15, 0.1])
    for row in range(n):
        X[row, X[row] == 7] = r.integers(0, 7)         # clear existing sevens
        pos = r.choice(20, size=counts[row], replace=False)
        X[row, pos] = 7
    y = (counts >= 3).astype(int)
    return X, y

X_train, y_train = make(6000, 471)
X_test, y_test = make(1500, 472)
print(f"X_train {X_train.shape}, positive rate {y_train.mean():.2f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.95

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

The reference solution below feeds bag-of-token counts to a small MLP. The decisive feature is the count of token 7. Building the transformer encoder from step 1 is left to you, and it reaches the same accuracy.

```python
import numpy as np
from sklearn.neural_network import MLPClassifier

def token_counts(X):
    return np.stack([np.bincount(row.astype(np.intp), minlength=10) for row in X])

Xtr = token_counts(X_train)
Xte = token_counts(X_test)

clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=800, random_state=0)
clf.fit(Xtr, y_train)
predictions = clf.predict(Xte)
```
